In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/post_cell_9.pickle

In [4]:
%%RecordEvent
%%time
### cell 10 ###

# Rewritten for cudf (avoid unstack and axis-1/div CPU fallbacks)

# 1. Top 11 countries by count (value_counts is GPU)
country_order = df_cell6['first_country'].value_counts().head(11).index

# 2. Compute the (country, type) counts in long form on GPU
counts = (
    df_cell6
    .groupby(['first_country', 'type'])
    .size()
    .reset_index(name='count')
)

# 3. Pivot to wide form WITHOUT unstack
movies = counts[counts['type'] == 'Movie'][['first_country', 'count']]
movies = movies.rename(columns={'count': 'Movie'})

tv = counts[counts['type'] == 'TV Show'][['first_country', 'count']]
tv = tv.rename(columns={'count': 'TV Show'})

# 4. Merge on GPU and reindex
data_q2q3 = movies.merge(tv, on='first_country', how='outer').fillna(0)
data_q2q3 = data_q2q3.set_index('first_country').loc[country_order]

# 5. Compute row sum with GPU addition
data_q2q3['sum'] = data_q2q3['Movie'] + data_q2q3['TV Show']

# 6. Compute ratios elementwise (GPU) and sort
#    Avoid DataFrame.div (CPU fallback) and axis=1 sum
data_q2q3_ratio = data_q2q3[['Movie', 'TV Show']].copy()
# elementwise division stays on GPU
data_q2q3_ratio['Movie']   = data_q2q3_ratio['Movie']   / data_q2q3['sum']
data_q2q3_ratio['TV Show'] = data_q2q3_ratio['TV Show'] / data_q2q3['sum']

# sort_values is GPU
data_q2q3_ratio = data_q2q3_ratio.sort_values('Movie', ascending=True)

CPU times: user 778 ms, sys: 255 ms, total: 1.03 s
Wall time: 1.02 s


In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_1.pickle